# RDR2 Case Study

## Defining Visualisation Tools

In [1]:
import pandas as pd

# Residue to prototype mapping

def add_residue_to_prototype_mapping(segment_assignments, prototype_assignments):
    pdb_id_segment_to_prototype = {}
    
    for _, row in prototype_assignments.iterrows():
        key = (row['pdb_id'], row['segment_k'])
        pdb_id_segment_to_prototype[key] = row['proto']
    
    out = segment_assignments.copy()

    def parse_and_map(row):
        cluster_ids = [
            int(x.strip())
            for x in row['cluster_ids'].split(',')
            if x.strip() != ''
        ]
        return [
            pdb_id_segment_to_prototype.get((row['pdb_id'], cid))
            for cid in cluster_ids
        ]

    out['protos'] = out.apply(parse_and_map, axis=1)
    return out

In [2]:
import colorsys
import random
from collections import defaultdict


def generate_proto_colors(proto_ids, seed=17):
    """Stable proto_id → HEX color mapping for py3Dmol"""
    unique_protos = sorted({p for p in proto_ids if p is not None})
    n = len(unique_protos)

    colors = []
    for i in range(n):
        hue = i / n
        saturation = 0.8 + (i % 2) * 0.2
        value = 0.8 + ((i // 2) % 2) * 0.2

        r, g, b = colorsys.hsv_to_rgb(hue, saturation, value)
        hex_color = f"#{int(r*255):02x}{int(g*255):02x}{int(b*255):02x}"
        colors.append(hex_color)

    random.seed(seed)
    random.shuffle(colors)

    return {
        proto: colors[i]
        for i, proto in enumerate(unique_protos)
    }


In [3]:
def group_residues_by_proto(residue_ids, proto_ids):
    """
    residue_ids: list[int]
    proto_ids:   list[int | str]
    """
    assert len(residue_ids) == len(proto_ids)

    proto_to_residues = defaultdict(list)
    for resi, proto in zip(residue_ids, proto_ids):
        proto_to_residues[proto].append(resi)

    return proto_to_residues


In [4]:
def get_residue_ids(df, pdb_id):
    row = df[df["pdb_id"] == pdb_id].iloc[0]
    residue_ids = [
            int(x.strip())
            for x in row['residue_ids'].split(',')
            if x.strip() != ''
        ]
    return residue_ids

In [5]:
def get_proto_ids(df, pdb_id):
    row = df[df["pdb_id"] == pdb_id].iloc[0]
    return row['protos']

In [6]:
from IPython.display import HTML, display

def show_colors_html(proto_to_color):
    html = "<div style='display:flex; flex-wrap:wrap;'>"
    for proto, color in proto_to_color.items():
        html += (
            f"<div style='width:120px; margin:4px; "
            f"background:{color}; color:black; "
            f"padding:6px; text-align:center; "
            f"border:1px solid #ccc;'>"
            f"{proto}<br>{color}</div>"
        )
    html += "</div>"
    display(HTML(html))

## Visualizing Interpro Domains

In [7]:
import json
with open("data/GeneOntology/test_interpro.json", "r") as f:
    interpro = json.load(f)

In [8]:
searched_pdb_id = '2RD2-A_A'
searched_pdb = searched_pdb_id.split("-")[0]
searched_chain = searched_pdb_id.split("_")[1]

In [9]:
import py3Dmol
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import interpro_proto_go_term_comparison as ipgtc

interpro_data = ipgtc.extract_interpro_intervals(interpro, searched_pdb, searched_chain)
view = py3Dmol.view(query=f'pdb:{searched_pdb}', width=800, height=600)

view.setStyle({}, {})
# 1. Default Style: Set everything to light gray first
view.setStyle({'chain': searched_chain}, {'cartoon': {'color': 'lightgray'}})

# 2. Generate unique colors for each distinct accession
unique_accessions = list(set(item['accession'] for item in interpro_data))
colors = list(mcolors.TABLEAU_COLORS.values()) # Using a standard color set
acc_to_color = {"IPR020058": "#ff7f0e", "IPR020059": "#1f77b4"}

# 3. Overlay the Interpro Annotations
for anno in interpro_data:
    if anno['accession'] not in ['IPR020058', 'IPR020059']:
        continue  # Skip if no accession is available
    res_range = f"{anno['start0']}-{anno['end0']}"
    
    view.addStyle(
        {'chain': searched_chain, 'resi': res_range},
        {'cartoon': {'color': acc_to_color[anno['accession']]}}
    )

# 4. View adjustments
view.zoomTo({'chain': searched_chain})
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [10]:
show_colors_html(acc_to_color)

## Visualising PUFFIN Unit Clusters

In [12]:
results_folder = "results_04_02"
model_name = "puffin_K64_v4"
model_k = "K1024"

In [13]:
import pandas as pd
prototype_assignments = pd.read_csv(f"{results_folder}/prototypes/{model_name}/{model_k}/assignments_test.csv")
segment_assignments = pd.read_csv(f"{results_folder}/segments/{model_name}/test/test_residue_assignments.csv")
segment_assignments_with_protos = add_residue_to_prototype_mapping(segment_assignments, prototype_assignments)
enrichment_df = pd.read_csv(f"{results_folder}/prototypes/{model_name}/{model_k}/eval/go_enrichment_train_all.csv")

In [14]:
segment_assignments_with_protos[segment_assignments_with_protos["pdb_id"] == searched_pdb_id]

,pdb_id,n_residues,residue_ids,cluster_ids,K,protos
293,2RD2-A_A,529,"8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,...","62,62,62,62,62,62,62,62,62,62,62,62,62,62,62,6...",64,"[563, 563, 563, 563, 563, 563, 563, 563, 563, ..."


In [15]:
from interpro_segment_iou_calc import match_interpro_and_protos, add_residue_to_segment_mapping


segment_assignments_final = add_residue_to_segment_mapping(
        segment_assignments_with_protos)

match_interpro_and_protos(
            interpro_json=interpro,
            segment_assignments_with_protos=segment_assignments_final,
            pdb_id_full=searched_pdb_id,
        )['interpro_to_proto']

[{'pdb_id': '2RD2-A_A',
  'ipr_accession': 'IPR000924',
  'ipr_name': 'Glutamyl/glutaminyl-tRNA synthetase',
  'ipr_type': 'family',
  'len_ipr_residues': 13,
  'best_proto': None,
  'best_iou': 0.0},
 {'pdb_id': '2RD2-A_A',
  'ipr_accession': 'IPR000924',
  'ipr_name': 'Glutamyl/glutaminyl-tRNA synthetase',
  'ipr_type': 'family',
  'len_ipr_residues': 12,
  'best_proto': None,
  'best_iou': 0.0},
 {'pdb_id': '2RD2-A_A',
  'ipr_accession': 'IPR000924',
  'ipr_name': 'Glutamyl/glutaminyl-tRNA synthetase',
  'ipr_type': 'family',
  'len_ipr_residues': 14,
  'best_proto': None,
  'best_iou': 0.0},
 {'pdb_id': '2RD2-A_A',
  'ipr_accession': 'IPR000924',
  'ipr_name': 'Glutamyl/glutaminyl-tRNA synthetase',
  'ipr_type': 'family',
  'len_ipr_residues': 11,
  'best_proto': None,
  'best_iou': 0.0},
 {'pdb_id': '2RD2-A_A',
  'ipr_accession': 'IPR001412',
  'ipr_name': 'Aminoacyl-tRNA synthetase, class I, conserved site',
  'ipr_type': 'conserved_site',
  'len_ipr_residues': 12,
  'best_proto'

In [21]:
import py3Dmol

# Create viewer
view = py3Dmol.view(query=f'pdb:{searched_pdb}', width=800, height=600)

view.setStyle({}, {})

view.setStyle(
    {'chain': searched_chain},
    {'cartoon': {'color': 'lightgray'}}
)

residue_ids = get_residue_ids(segment_assignments_with_protos, pdb_id=searched_pdb_id)
proto_ids = get_proto_ids(segment_assignments_with_protos, pdb_id=searched_pdb_id)

proto_to_residues = group_residues_by_proto(residue_ids, proto_ids)
proto_to_color = {178: "#cc2828", 496: "#3232ff", 563: "#00cc00"}
for proto, residues in proto_to_residues.items():
    if proto != None:
        view.addStyle(
            {
                'chain': searched_chain,
                'resi': residues
            },
            {
                'cartoon': {'color': proto_to_color[proto]}
            }
        )
view.zoomTo({'chain': searched_chain})
view


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [22]:
show_colors_html(proto_to_color)

## Visualising ESM K-Means Unit Clusters

In [23]:
results_folder = "results_04_02"
model_name = "esm_kmeans"
model_k = "K1024"

In [24]:
import pandas as pd
prototype_assignments = pd.read_csv(f"{results_folder}/prototypes/{model_name}/{model_k}/assignments_test.csv")
segment_assignments = pd.read_csv(f"{results_folder}/segments/{model_name}/test/test_residue_assignments.csv")
segment_assignments_with_protos = add_residue_to_prototype_mapping(segment_assignments, prototype_assignments)
enrichment_df = pd.read_csv(f"{results_folder}/prototypes/{model_name}/{model_k}/eval/go_enrichment_train_all.csv")

In [25]:
segment_assignments_with_protos[segment_assignments_with_protos["pdb_id"] == searched_pdb_id]

,pdb_id,n_residues,residue_ids,cluster_ids,K,protos
293,2RD2-A_A,529,"8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,...","10,10,10,10,10,10,10,10,10,10,10,10,10,10,61,1...",64,"[421, 421, 421, 421, 421, 421, 421, 421, 421, ..."


In [27]:
from interpro_segment_iou_calc import match_interpro_and_protos, add_residue_to_segment_mapping


segment_assignments_final = add_residue_to_segment_mapping(
        segment_assignments_with_protos)

match_interpro_and_protos(
            interpro_json=interpro,
            segment_assignments_with_protos=segment_assignments_final,
            pdb_id_full=searched_pdb_id,
        )['interpro_to_proto']

[{'pdb_id': '2RD2-A_A',
  'ipr_accession': 'IPR000924',
  'ipr_name': 'Glutamyl/glutaminyl-tRNA synthetase',
  'ipr_type': 'family',
  'len_ipr_residues': 13,
  'best_proto': None,
  'best_iou': 0.0},
 {'pdb_id': '2RD2-A_A',
  'ipr_accession': 'IPR000924',
  'ipr_name': 'Glutamyl/glutaminyl-tRNA synthetase',
  'ipr_type': 'family',
  'len_ipr_residues': 12,
  'best_proto': None,
  'best_iou': 0.0},
 {'pdb_id': '2RD2-A_A',
  'ipr_accession': 'IPR000924',
  'ipr_name': 'Glutamyl/glutaminyl-tRNA synthetase',
  'ipr_type': 'family',
  'len_ipr_residues': 14,
  'best_proto': None,
  'best_iou': 0.0},
 {'pdb_id': '2RD2-A_A',
  'ipr_accession': 'IPR000924',
  'ipr_name': 'Glutamyl/glutaminyl-tRNA synthetase',
  'ipr_type': 'family',
  'len_ipr_residues': 11,
  'best_proto': None,
  'best_iou': 0.0},
 {'pdb_id': '2RD2-A_A',
  'ipr_accession': 'IPR001412',
  'ipr_name': 'Aminoacyl-tRNA synthetase, class I, conserved site',
  'ipr_type': 'conserved_site',
  'len_ipr_residues': 12,
  'best_proto'

In [29]:
import py3Dmol

# Create viewer
view = py3Dmol.view(query=f'pdb:{searched_pdb}', width=800, height=600)

view.setStyle({}, {})

view.setStyle(
    {'chain': searched_chain},
    {'cartoon': {'color': 'lightgray'}}
)

residue_ids = get_residue_ids(segment_assignments_with_protos, pdb_id=searched_pdb_id)
proto_ids = get_proto_ids(segment_assignments_with_protos, pdb_id=searched_pdb_id)

proto_to_residues = group_residues_by_proto(residue_ids, proto_ids)
proto_to_color = generate_proto_colors(proto_ids)
for proto, residues in proto_to_residues.items():
    """if proto not in [855]:
        continue"""
    if proto != None:
        view.addStyle(
            {
                'chain': searched_chain,
                'resi': residues
            },
            {
                'cartoon': {'color': proto_to_color[proto]}
            }
        )
view.zoomTo({'chain': searched_chain})
view


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [30]:
show_colors_html(proto_to_color)